## Creating a DF from cleanse data present in silver layer

In [2]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import *

StatementMeta(, d77feb21-8a93-425c-ab17-255d533f4be6, 5, Finished, Available, Finished, False)

In [15]:
df = spark.sql("""
SELECT * FROM Silver.Cleanse_Data.cleaned_stock_data
""")
df.show()

StatementMeta(, d77feb21-8a93-425c-ab17-255d533f4be6, 152, Finished, Available, Finished, False)

+-------------------+-----------+--------+--------+--------+--------+----------+----+-----+-----------+------------+
|          timestamp|     ticker|    open|    high|     low|   close|    volume|year|month|day_of_week|day_of_month|
+-------------------+-----------+--------+--------+--------+--------+----------+----+-----+-----------+------------+
|2023-02-23 00:00:00|        DIS|   98.36|   98.94|   96.58|   98.63|  10974800|2023|    2|          5|          23|
|2023-02-06 00:00:00|       AAPL|  150.02|  150.54|  148.26|   149.2|  69858300|2023|    2|          2|           6|
|2023-02-15 00:00:00|         KO|   54.01|   54.04|   53.41|   53.73|  13171400|2023|    2|          4|          15|
|2023-02-17 00:00:00|    INFY.NS| 1430.94| 1431.83| 1414.33| 1421.78|   2857734|2023|    2|          6|          17|
|2023-02-17 00:00:00|       NVDA|   21.58|   21.69|   20.93|   21.34| 465888000|2023|    2|          6|          17|
|2023-02-03 00:00:00|         KO|   54.42|   54.48|   53.41|   5

### **Table 1: dim_stock**

In [16]:
unique_stocks = df.select(("ticker")).distinct()

StatementMeta(, d77feb21-8a93-425c-ab17-255d533f4be6, 154, Finished, Available, Finished, False)

In [17]:
window = Window.orderBy("ticker")
id_unique_stocks= unique_stocks.select(row_number().over(window).alias("stock_id"), col("ticker"))
id_unique_stocks.show(35)

StatementMeta(, d77feb21-8a93-425c-ab17-255d533f4be6, 155, Finished, Available, Finished, False)

+--------+------------+
|stock_id|      ticker|
+--------+------------+
|       1|        AAPL|
|       2|        ADBE|
|       3|         AMD|
|       4|        AMZN|
|       5|         BAC|
|       6|     BTC-USD|
|       7|        COST|
|       8|         CRM|
|       9|         DIS|
|      10|     ETH-USD|
|      11|       GOOGL|
|      12| HDFCBANK.NS|
|      13|         IBM|
|      14|ICICIBANK.NS|
|      15|     INFY.NS|
|      16|        INTC|
|      17|      ITC.NS|
|      18|         JPM|
|      19|          KO|
|      20|         MCD|
|      21|        META|
|      22|        MSFT|
|      23|        NFLX|
|      24|        NVDA|
|      25|        ORCL|
|      26|         PEP|
|      27| RELIANCE.NS|
|      28|     SBIN.NS|
|      29|      TCS.NS|
|      30|        TSLA|
|      31|         WMT|
|      32|      ^BSESN|
|      33|       ^GSPC|
|      34|       ^IXIC|
|      35|       ^NSEI|
+--------+------------+



In [18]:
asset_metadata = [
    ("AAPL", "Apple", "Equity", "US", "NASDAQ"),
    ("ADBE", "Adobe", "Equity", "US", "NASDAQ"),
    ("AMD", "Advanced Micro Devices", "Equity", "US", "NASDAQ"),
    ("AMZN", "Amazon", "Equity", "US", "NASDAQ"),
    ("BAC", "Bank of America", "Equity", "US", "NYSE"),
    ("BTC-USD", "Bitcoin", "Crypto", "Global", "Crypto"),
    ("COST", "Costco Wholesale", "Equity", "US", "NASDAQ"),
    ("CRM", "Salesforce", "Equity", "US", "NYSE"),
    ("DIS", "Walt Disney", "Equity", "US", "NYSE"),
    ("ETH-USD", "Ethereum", "Crypto", "Global", "Crypto"),
    ("GOOGL", "Alphabet", "Equity", "US", "NASDAQ"),
    ("HDFCBANK.NS", "HDFC Bank", "Equity", "India", "NSE"),
    ("IBM", "IBM", "Equity", "US", "NYSE"),
    ("ICICIBANK.NS", "ICICI Bank", "Equity", "India", "NSE"),
    ("INFY.NS", "Infosys", "Equity", "India", "NSE"),
    ("INTC", "Intel", "Equity", "US", "NASDAQ"),
    ("ITC.NS", "ITC", "Equity", "India", "NSE"),
    ("JPM", "JPMorgan Chase", "Equity", "US", "NYSE"),
    ("KO", "The Coca-Cola Company", "Equity", "US", "NYSE"),
    ("MCD", "McDonald's", "Equity", "US", "NYSE"),
    ("META", "Meta Platforms", "Equity", "US", "NASDAQ"),
    ("MSFT", "Microsoft", "Equity", "US", "NASDAQ"),
    ("NFLX", "Netflix", "Equity", "US", "NASDAQ"),
    ("NVDA", "NVIDIA", "Equity", "US", "NASDAQ"),
    ("ORCL", "Oracle", "Equity", "US", "NYSE"),
    ("PEP", "PepsiCo", "Equity", "US", "NASDAQ"),
    ("RELIANCE.NS", "Reliance Industries", "Equity", "India", "NSE"),
    ("SBIN.NS", "State Bank of India", "Equity", "India", "NSE"),
    ("TCS.NS", "Tata Consultancy Services", "Equity", "India", "NSE"),
    ("TSLA", "Tesla", "Equity", "US", "NASDAQ"),
    ("WMT", "Walmart", "Equity", "US", "NYSE"),
    ("^BSESN", "BSE Sensex", "Index", "India", "BSE"),
    ("^GSPC", "S&P 500", "Index", "US", "S&P"),
    ("^IXIC", "NASDAQ Composite", "Index", "US", "NASDAQ"),
    ("^NSEI", "NIFTY 50", "Index", "India", "NSE")
]

metadata_df = spark.createDataFrame(
    asset_metadata,
    ["ticker", "asset_name", "asset_type", "market", "exchange"]
)

StatementMeta(, d77feb21-8a93-425c-ab17-255d533f4be6, 157, Finished, Available, Finished, False)

In [19]:
dim_stock = id_unique_stocks.withColumnRenamed("asset_id","stock_id")\
                           .join(metadata_df, on="ticker", how="left").orderBy("stock_id")
dim_stock.show(35, truncate=False)                           

StatementMeta(, d77feb21-8a93-425c-ab17-255d533f4be6, 158, Finished, Available, Finished, False)

+------------+--------+-------------------------+----------+------+--------+
|ticker      |stock_id|asset_name               |asset_type|market|exchange|
+------------+--------+-------------------------+----------+------+--------+
|AAPL        |1       |Apple                    |Equity    |US    |NASDAQ  |
|ADBE        |2       |Adobe                    |Equity    |US    |NASDAQ  |
|AMD         |3       |Advanced Micro Devices   |Equity    |US    |NASDAQ  |
|AMZN        |4       |Amazon                   |Equity    |US    |NASDAQ  |
|BAC         |5       |Bank of America          |Equity    |US    |NYSE    |
|BTC-USD     |6       |Bitcoin                  |Crypto    |Global|Crypto  |
|COST        |7       |Costco Wholesale         |Equity    |US    |NASDAQ  |
|CRM         |8       |Salesforce               |Equity    |US    |NYSE    |
|DIS         |9       |Walt Disney              |Equity    |US    |NYSE    |
|ETH-USD     |10      |Ethereum                 |Crypto    |Global|Crypto  |

### **Table 2: dim_date**

In [9]:
%%pyspark

dim_date = spark.sql("""
SELECT 
REPLACE(CAST(timestamp AS DATE), '-', '') AS date_id,
cast(timestamp AS DATE), day_of_month AS day, 
month,
year
FROM Silver.Cleanse_Data.cleaned_stock_data
""").distinct().orderBy("timestamp")
dim_date.show(1)


StatementMeta(, d77feb21-8a93-425c-ab17-255d533f4be6, 99, Finished, Available, Finished, False)

+--------+----------+---+-----+----+
| date_id| timestamp|day|month|year|
+--------+----------+---+-----+----+
|20210807|2021-08-07|  7|    8|2021|
+--------+----------+---+-----+----+
only showing top 1 row



### Created a table from dim_stock

In [20]:
dim_stock.createOrReplaceTempView("dim_stock_table")

StatementMeta(, d77feb21-8a93-425c-ab17-255d533f4be6, 159, Finished, Available, Finished, False)

In [21]:
fact_stock_prices = spark.sql("""
SELECT replace(CAST(timestamp AS DATE),'-','') AS date_id,
d.stock_id,
timestamp,
open, high, low, close, volume
FROM Silver.Cleanse_Data.cleaned_stock_data as s JOIN dim_stock_table AS d
on s.ticker= d.ticker
ORDER BY date_id, stock_id

""")
fact_stock_prices.show(2)

StatementMeta(, d77feb21-8a93-425c-ab17-255d533f4be6, 160, Finished, Available, Finished, False)

+--------+--------+-------------------+-------+--------+--------+-------+-----------+
| date_id|stock_id|          timestamp|   open|    high|     low|  close|     volume|
+--------+--------+-------------------+-------+--------+--------+-------+-----------+
|20210807|       6|2021-08-07 00:00:00|42832.8|44689.86|42618.57|44555.8|40030862141|
|20210807|      10|2021-08-07 00:00:00|2891.71| 3170.23| 2868.54|3157.24|33081467129|
+--------+--------+-------------------+-------+--------+--------+-------+-----------+
only showing top 2 rows



In [29]:
dim_stock.show(5)

StatementMeta(, 654f2bc4-071c-44be-a1da-c06a601c7371, 61, Finished, Available, Finished, False)

+------+--------+--------------------+----------+------+--------+
|ticker|stock_id|          asset_name|asset_type|market|exchange|
+------+--------+--------------------+----------+------+--------+
|  AAPL|       1|               Apple|    Equity|    US|  NASDAQ|
|  ADBE|       2|               Adobe|    Equity|    US|  NASDAQ|
|   AMD|       3|Advanced Micro De...|    Equity|    US|  NASDAQ|
|  AMZN|       4|              Amazon|    Equity|    US|  NASDAQ|
|   BAC|       5|     Bank of America|    Equity|    US|    NYSE|
+------+--------+--------------------+----------+------+--------+
only showing top 5 rows



In [33]:
dim_date.show(5)

StatementMeta(, 654f2bc4-071c-44be-a1da-c06a601c7371, 71, Finished, Available, Finished, False)

+--------+----------+---+-----+----+
| date_id| timestamp|day|month|year|
+--------+----------+---+-----+----+
|20210807|2021-08-07|  7|    8|2021|
|20210808|2021-08-08|  8|    8|2021|
|20210809|2021-08-09|  9|    8|2021|
|20210810|2021-08-10| 10|    8|2021|
|20210811|2021-08-11| 11|    8|2021|
+--------+----------+---+-----+----+
only showing top 5 rows



# **Saving fact_stock_prices as delta table in gold lakehouse**

In [23]:
fact_stock_prices.write.format("delta") \
                       .mode("overwrite")\
                       .saveAsTable("Gold.price.fact_stock_prices")

StatementMeta(, d77feb21-8a93-425c-ab17-255d533f4be6, 169, Finished, Available, Finished, False)

# **Saving dim_date as delta table in gold lakehouse**

In [35]:
dim_date.write.format("delta")\
              .mode("overwrite")\
              .saveAsTable("Gold.date.dim_date")

StatementMeta(, 654f2bc4-071c-44be-a1da-c06a601c7371, 74, Finished, Available, Finished, False)

# **Saving dim_stock as delta table in gold lakehouse**

In [37]:
dim_stock.write.format("delta")\
              .mode("overwrite")\
              .saveAsTable("Gold.stock.dim_stock")

StatementMeta(, 654f2bc4-071c-44be-a1da-c06a601c7371, 76, Finished, Available, Finished, False)